# Chapter 4: From Gradient Boosting to XGBoost

# 9.0 - XGBoost for Time-series classification problem using a **Big** DataSet

## 🪐 9.1 - Exoplanet Detection Dataset: Structure and Challenges


---
### 📊 Dataset Overview

This dataset focuses on detecting **exoplanets** (planets outside our solar system) based on changes in **light flux** (perceived brightness of a star) from stars over time.

* **Rows:** 5,087
  Each row represents a **unique star observation**.

* **Columns:** 3,189
  Each column captures **light flux** at a specific **time step** in the star’s life cycle, forming a high-resolution **light curve**.

> 🧠 In total, this creates **\~1.5 million data points** (5,087 × 3,189) — effectively a time-series dataset for each star.

When training models like Gradient Boosting or XGBoost with a baseline of **100 trees**, the model processes:

* **\~150 million data points** (1.5 million × 100), highlighting the need for **efficient algorithms**.

---

### 🎯 Target Variable: Exoplanet Presence

The **target column** indicates whether a star **hosts an exoplanet**:

* **2 = Exoplanet present**
* **1 = No exoplanet**

However, exoplanets are **rare**, so the dataset is **highly imbalanced**:

* Most stars do **not** have exoplanets (majority class = 1)
* Only a small fraction are positive cases (minority class = 2)

---

### ⚠️ Key Challenges

1. **High Dimensionality:**

   * With over 3,000 time-based features per star, the model needs to handle **complex temporal patterns**.

2. **Data Volume:**

   * The large number of data points makes **training resource-intensive**, requiring optimized models like **XGBoost**.

3. **Class Imbalance:**

   * Standard accuracy may be misleading.
   * Requires careful handling through:

     * **Balanced evaluation metrics** (e.g., F1-score, AUC)
     * **Class weighting or resampling**
     * **Specialized algorithms** tuned for rare-event prediction

---

## 9.2 - Preprocessing the exoplanet dataset

In [7]:
# Import pandas and numpy
import pandas as pd

# Silence warnings
import warnings
warnings.filterwarnings('ignore')

In [8]:
df = pd.read_csv('exoplanets.csv')
df.head()

,LABEL,FLUX.1,FLUX.2,FLUX.3,FLUX.4,FLUX.5,FLUX.6,FLUX.7,FLUX.8,FLUX.9,...,FLUX.3188,FLUX.3189,FLUX.3190,FLUX.3191,FLUX.3192,FLUX.3193,FLUX.3194,FLUX.3195,FLUX.3196,FLUX.3197
0,2,93.85,83.81,20.10,-26.98,-39.56,-124.71,-135.18,-96.27,-79.89,...,-78.07,-102.15,-102.15,25.13,48.57,92.54,39.32,61.42,5.08,-39.54
1,2,-38.88,-33.83,-58.54,-40.09,-79.31,-72.81,-86.55,-85.33,-83.97,...,-3.28,-32.21,-32.21,-24.89,-4.86,0.76,-11.70,6.46,16.00,19.93
2,2,532.64,535.92,513.73,496.92,456.45,466.00,464.50,486.39,436.56,...,-71.69,13.31,13.31,-29.89,-20.88,5.06,-11.80,-28.91,-70.02,-96.67
3,2,326.52,347.39,302.35,298.13,317.74,312.70,322.33,311.31,312.42,...,5.71,-3.73,-3.73,30.05,20.03,-12.67,-8.77,-17.31,-17.35,13.98
4,2,-1107.21,-1112.59,-1118.95,-1095.10,-1057.55,-1034.48,-998.34,-1022.71,-989.57,...,-594.37,-401.66,-401.66,-357.24,-443.76,-438.54,-399.71,-384.65,-411.79,-510.54


Ideally, **classification labels should be 0 and 1** for binary classification, especially with scikit-learn and XGBoost classifiers.

---

### 🔍 Your Situation

* Your dataset has labels **1** and **2**.
* Most classifiers (like `GradientBoostingClassifier`, `XGBClassifier`, `LogisticRegression`) interpret:

  * **0 = Negative class**
  * **1 = Positive class**

Using **1 and 2** won’t break the model, but it may:

* Lead to **confusing evaluation results** (e.g., precision, confusion matrix)
* Mislabel **which class is considered "positive"**
* Cause issues in downstream metrics like `classification_report()` or `roc_auc_score()`, which assume 0/1 for binary classification

---

### ✅ Best Practice: Convert 1/2 → 0/1

```python
# Convert label column from 1,2 to 0,1
df['LABEL'] = df['LABEL'] - 1
```

This transforms:

* 1 → 0 (no exoplanet)
* 2 → 1 (exoplanet)

---

### 🧠 Why This Matters

* Many ML tools (especially ROC curves and AUC) assume 1 is the "positive" class.
* Keeping labels as 0 and 1 helps ensure **metrics are interpreted correctly**.

---


✅ Python Script to Convert LABEL from 1/2 → 0/1

In [19]:
import pandas as pd

# Load the dataset
df = pd.read_csv("exoplanets.csv")

# Print original unique label values
print("Original LABEL values:", df['LABEL'].unique())

# Convert LABEL column: 1 → 0, 2 → 1
df['LABEL'] = df['LABEL'] - 1

# Print new unique label values for confirmation
print("Converted LABEL values:", df['LABEL'].unique())

# Save the updated DataFrame to a new CSV file
df.to_csv("exoplanets_binary_labels.csv", index=False)

print("✅ Dataset saved with binary labels (0 and 1) as 'exoplanets_binary_labels.csv'")


Original LABEL values: [2 1]
Converted LABEL values: [1 0]
✅ Dataset saved with binary labels (0 and 1) as 'exoplanets_binary_labels.csv'


In [20]:
df = pd.read_csv('exoplanets_binary_labels.csv')
df.head()

,LABEL,FLUX.1,FLUX.2,FLUX.3,FLUX.4,FLUX.5,FLUX.6,FLUX.7,FLUX.8,FLUX.9,...,FLUX.3188,FLUX.3189,FLUX.3190,FLUX.3191,FLUX.3192,FLUX.3193,FLUX.3194,FLUX.3195,FLUX.3196,FLUX.3197
0,1,93.85,83.81,20.10,-26.98,-39.56,-124.71,-135.18,-96.27,-79.89,...,-78.07,-102.15,-102.15,25.13,48.57,92.54,39.32,61.42,5.08,-39.54
1,1,-38.88,-33.83,-58.54,-40.09,-79.31,-72.81,-86.55,-85.33,-83.97,...,-3.28,-32.21,-32.21,-24.89,-4.86,0.76,-11.70,6.46,16.00,19.93
2,1,532.64,535.92,513.73,496.92,456.45,466.00,464.50,486.39,436.56,...,-71.69,13.31,13.31,-29.89,-20.88,5.06,-11.80,-28.91,-70.02,-96.67
3,1,326.52,347.39,302.35,298.13,317.74,312.70,322.33,311.31,312.42,...,5.71,-3.73,-3.73,30.05,20.03,-12.67,-8.77,-17.31,-17.35,13.98
4,1,-1107.21,-1112.59,-1118.95,-1095.10,-1057.55,-1034.48,-998.34,-1022.71,-989.57,...,-594.37,-401.66,-401.66,-357.24,-443.76,-438.54,-399.71,-384.65,-411.79,-510.54


In [21]:
df.shape

(5087, 3198)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5087 entries, 0 to 5086
Columns: 3198 entries, LABEL to FLUX.3197
dtypes: float64(3197), int64(1)
memory usage: 124.1 MB


---

### 📌 Summary:

| Part               | Meaning                               |
| ------------------ | ------------------------------------- |
| **5087 rows**      | Each row = a star's observation       |
| **3198 columns**   | `LABEL` + 3,197 flux values           |
| **float64, int64** | Flux values are floats, target is int |
| **124.1 MB**       | Total memory footprint in RAM         |

---

### 🎯 Interpretation

* **Use Case**: This is a **time-series classification** problem — you're using the light flux signal (features) to predict whether a star hosts an exoplanet (`LABEL`).
* **High dimensionality**: 3,197 features per observation makes the dataset **wide**, which may require dimensionality reduction or regularization.

---

In [23]:
# Check for missing (null/NaN) values in the entire DataFrame

# Step 1: df.isnull()
# This returns a DataFrame of the same shape as df
# Each cell will contain True if the value is missing (NaN), otherwise False

# Step 2: .sum()
# When called on a DataFrame of True/False values:
# - It treats True as 1 and False as 0
# - The first .sum() adds up missing values *column-wise*
#   → Result: a Series showing number of missing values per column

# Step 3: .sum() again
# Adds up all the column-wise sums to get the *total number* of missing values in the entire DataFrame

df.isnull().sum().sum()

np.int64(0)

The output reveals that there are no null values.

In [24]:
# Import the train_test_split function from scikit-learn
# This function helps us split the data into training and testing sets
from sklearn.model_selection import train_test_split

In [25]:
# Split data into X and y
X = df.iloc[:,1:]
y = df.iloc[:,0]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2)

In [26]:
print(f"X_train.shape: {X_train.shape}")
print(f"X_test.shape: {X_test.shape}")
print(f"y_train.shape: {y_train.shape}")
print(f"y_test.shape: {y_test.shape}")

X_train.shape: (3815, 3197)
X_test.shape: (1272, 3197)
y_train.shape: (3815,)
y_test.shape: (1272,)


In [27]:
from sklearn.ensemble import GradientBoostingClassifier
# Import XGBRegressor
from xgboost import XGBClassifier

# Import accuracy_score
from sklearn.metrics import accuracy_score

In [28]:
import time
start = time.time()

df.info()

end = time.time()
elapsed = end - start

print('\nRun Time: ' + str(elapsed) + ' seconds.')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5087 entries, 0 to 5086
Columns: 3198 entries, LABEL to FLUX.3197
dtypes: float64(3197), int64(1)
memory usage: 124.1 MB

Run Time: 0.03630971908569336 seconds.


## Comparing speed between **GradientBoostingClassifier** and **XGBoostClassifier**

In [29]:
# Import required libraries
import time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Start measuring execution time
start = time.time()

# Initialize a Gradient Boosting Classifier
# - n_estimators=100 → Number of trees to build in the ensemble
# - max_depth=2 → Shallow trees (weak learners) to prevent overfitting
# - random_state=2 → Ensures reproducibility of results
gbr = GradientBoostingClassifier(n_estimators=100, max_depth=2, random_state=2)

# Fit the model on the training data
gbr.fit(X_train, y_train)

# Predict the labels for the test set
y_pred = gbr.predict(X_test)

# Evaluate model performance using accuracy
# - accuracy_score compares predicted labels to actual test labels
score = accuracy_score(y_pred, y_test)

# Print the accuracy as a string
print('Score: ' + str(score))

# Stop timing and calculate how long the model took to run
end = time.time()
elapsed = end - start

# Print the total run time in seconds
print('Run Time: ' + str(elapsed) + ' seconds')


Score: 0.9874213836477987
Run Time: 416.13074946403503 seconds


While a score of 98.7% percent is usually outstanding for accuracy, this is not the case with **imbalanced datasets**

In [30]:
# Import required libraries
import time
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Start measuring execution time
start = time.time()

# Instantiate the XGBClassifier (XGBoost for classification tasks)
# - n_estimators=100 → Number of boosting rounds (trees)
# - max_depth=2 → Depth of each tree (shallow trees to avoid overfitting)
# - random_state=2 → Seed for reproducibility
xg_reg = XGBClassifier(n_estimators=100, max_depth=2, random_state=2)

# Fit the model on the training data
# X_train contains features; y_train contains labels (0 or 1)
xg_reg.fit(X_train, y_train)

# Predict labels on the test set
# y_pred contains predicted values for X_test
y_pred = xg_reg.predict(X_test)

# Evaluate the model's performance using accuracy
# Compares predicted labels (y_pred) to actual labels (y_test)
score = accuracy_score(y_pred, y_test)

# Print the accuracy score
print('Score: ' + str(score))

# Stop the timer and calculate total execution time
end = time.time()
elapsed = end - start

# Print how long the training and prediction took
print('Run Time: ' + str(elapsed) + ' seconds')


Score: 0.9913522012578616
Run Time: 25.011899709701538 seconds


Gradient Boosting Run Time: 416.13 seconds

XGBoost Run Time: 25.012 seconds